# Gap imputation — the counterfactual reframe (bidirectional, subspace)

The demand simulation recast as **constrained subspace imputation** (supervisor's
reframe). NOT step-ahead forecasting: we know the data on both sides of the
11:00–14:00 window, and inside it we know the *totals* (net_demand, demand, price,
calendar) at every step — only the 6-source **breakdown** is hidden.

**Why the 2:05 seam disappears:** the old approach ran one-directional through the
window and drifted, then snapped back to the real 14:00 value (that snap = the ramp
violation). Here a **bidirectional** model sees the 14:00 boundary, and a **two-sided
ramp tube** pins the fill to it — every consecutive pair incl. both seams is
ramp-feasible by construction.

**The subspace (verified):** `net_demand = SIGN·sources` to 0.000 MW, so inside the
gap the model is handed the exact *sum* of the six hidden numbers and imputes the
5-dim *split* — constrained by both boundaries + ramp + box + balance.

Pipeline & literature: `imputation/README.md`, `constraints/lit_review.md` themes 8–9.


In [ ]:
REPO = "github.com/nm-quan/energy_modelling.git"
TOKEN = ""   # PRIVATE repo: fine-grained READ-ONLY token (Contents: read)
BRANCH = "claude/model-bottlenecks-constraints-gb1aoj"
import os
url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.exists("energy_modelling"): !git clone -q --branch $BRANCH $url
%cd energy_modelling
!git pull -q
!nvidia-smi -L


## 1 · The bar to beat — interpolation baseline

Mask the real 11–14 window, fill by linear interpolation between boundaries, score
reconstruction WAPE vs measured truth (answer key exists — we hid real data).


In [ ]:
!python imputation/baseline.py
from IPython.display import Markdown, display
display(Markdown(open("imputation/results/baselines.md").read()))


## 2 · Sanity — the two-sided ramp tube is feasible incl. both seams


In [ ]:
!python imputation/constraints.py
!python imputation/gap_data.py


## 3 · Train the bi-LSTM imputer (5 years of masked windows, early-stopped) — **run this cell yourself (GPU)**

Random-position masked gaps over the 394k-row train flat; loss = masked
reconstruction + soft balance. **Early stopping is leak-free and task-matched:**
best epoch is chosen on a *midday-matched* validation set — gaps pinned to
11:00–14:00 in the held-out val split, exactly the test task — and the test set is
scored **once**, after training. (History: v1 selected epochs on test = leakage;
v2 on random-position val gaps = no leak but easier than midday, so val looked
better than test. Both fixed — see `gap_data.sample_val_midday_windows`.)

`--perturb` stays **off**: scaling demand while keeping the real breakdown as the
target teaches the model to *ignore* demand (we measured it). Responsiveness comes
from 5 years of natural demand variation + the balance term/projection.

Writes `results/bilstm_imputer.pt` + `results/bilstm_recon.json` (projected
reconstruction WAPE on the 186 real test 11–14 days).

In [ ]:
!python imputation/train.py --epochs 100 --patience 10 --context 48
display(Markdown("**Trained reconstruction (projected, 186 real test days):**"))
import json
print(json.dumps(json.load(open("imputation/results/bilstm_recon.json")), indent=2))

## 4 · The counterfactual — placebo + +10% demand response

placebo (g=0): measured response must be ≈0. scenario (+10%): feed the raised demand
into the gap's known subspace, re-impute, project — does the fleet deliver the extra
load (capture≈1), and stay feasible incl. both seams?


In [ ]:
!python imputation/scenario_eval.py --g 10


## 5 · What to conclude

- The bar is the interp baseline, macro **0.345**. If the trained model's macro WAPE
  lands **below 0.345** while constraint-clean, the learned imputer earns its keep.
  Honest result so far (100-epoch CPU run, val-early-stopped): **0.440 — it does
  NOT beat interpolation**; batteries dominate the miss (arbitrage, not pattern).
  In that case the honest simulator is **interpolation + hard projection**, which
  mirrors the forecasting track (persistence + anchor beat the neural models).
- **Placebo response ≈ 0, +10% capture ≈ 0.94, 0 ramp violations incl. both seams,
  0 SOC-infeasible days** ⇒ the pipeline responds to demand, feasibly, with the
  2:05 problem gone *by construction* (the projection guarantees feasibility for
  any fill).
- Next: GRIN (graph across channels) engine; landing-strip right-boundary for the
  energy-non-neutral case; uncertainty intervals (diffusion). See README.

## 6 · Deliverable — 4-day stacked dispatch graph (imputed gaps, +10%)

Mirrors `demand_simulation/study_stack_4day.py`: 4 consecutive test days, each
11:00–14:00 gap imputed. Three panels — actual / base fill / +10% fill. The stack
is **continuous across every gold edge** (no 2:05 seam), and in the bottom panel it
rises to the raised demand line inside each gap.

In [ ]:
!python imputation/stack_gap.py --days 4 --g 10
from IPython.display import Image, display
display(Image("imputation/results/figure/stack_gap_4day_g10.png"))